# Sri Lanka District Flood Risk — Multi-Step LSTM Early Warning System

---

## Project Overview

Flooding remains one of the most destructive and recurring natural hazards in Sri Lanka, causing widespread displacement, agricultural loss, and infrastructure damage across coastal lowlands, river plains, and monsoon-exposed districts. Early warning systems that can forecast flood risk several days ahead provide critical lead time for emergency response, resource pre-positioning, and community evacuation planning.

This notebook develops a **Multi-Day Ahead Flood Risk Early Warning System** using a deep learning sequence model. The model receives a window of historical climatological and hydrological observations for any district and forecasts the **flood risk score** for the next **T+1 through T+7 days**, enabling a full one-week lookahead horizon.

---

## Dataset

| Attribute | Detail |
|---|---|
| Source | `sri_lanka_flood_risk_modeled.csv` |
| Period | 1 January 2015 to 31 December 2024 |
| Spatial Coverage | 25 districts across 9 provinces |
| Climatic Zones | Dry, Wet, Intermediate |
| Granularity | Daily per district |
| Total Records | 91,325 |
| Missing Values | None |

---

## Methodology

The end-to-end pipeline follows this structure:

1. **Data Ingestion and Validation** — load, parse, and sanity-check the raw dataset
2. **Exploratory Data Analysis** — understand temporal patterns, spatial risk distribution, and feature correlations
3. **Feature Engineering** — construct lag features, rolling aggregates, momentum signals, and cyclical time encodings
4. **Preprocessing** — time-based train/validation/test split, min-max normalization fit only on training data
5. **Sequence Construction** — sliding window over each district time series to form input-output sequence pairs
6. **Model Architecture** — a global Bidirectional LSTM followed by a stacked LSTM, trained across all 25 districts simultaneously
7. **Training** — early stopping, learning rate scheduling, and gradient clipping for stable convergence
8. **Evaluation** — RMSE, MAE, R2 and MAPE per overall split and per forecast horizon
9. **Model Persistence** — save the trained model weights, scalers, feature manifest, and structured knowledge artifacts

---

## 1. Environment Setup

All required libraries are imported here. The notebook depends on PyTorch for deep learning, scikit-learn for preprocessing and metrics, and matplotlib/seaborn for visualization. Random seeds are fixed across NumPy, Python built-in random, and PyTorch to ensure full reproducibility of results.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import pickle
import random
import math
from datetime import datetime
from typing import Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False
})

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
print(f'Random seed     : {SEED}')

---

## 2. Configuration

All hyperparameters and path constants are centralized here. This practice separates configuration from logic, making it straightforward to rerun experiments with different settings without touching any downstream cell.

In [ ]:
DATASET_PATH  = '../Dataset/sri_lanka_flood_risk_modeled.csv'
ARTIFACTS_DIR = 'artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

CONFIG = {
    'sequence_length' : 30,
    'forecast_horizon': 7,
    'batch_size'      : 512,
    'epochs'          : 120,
    'learning_rate'   : 1e-3,
    'weight_decay'    : 1e-4,
    'dropout_rate'    : 0.25,
    'lstm_hidden_1'   : 128,
    'lstm_hidden_2'   : 64,
    'dense_hidden'    : 64,
    'patience'        : 15,
    'lr_patience'     : 7,
    'lr_factor'       : 0.5,
    'min_lr'          : 1e-6,
    'grad_clip'       : 1.0,
    'val_year_start'  : 2022,
    'test_year_start' : 2023,
}

TARGET_COL = 'flood_risk_score'

BASE_FEATURES = [
    'precipitation_sum',
    'rain_sum',
    'soil_moisture_0_to_7cm_mean',
    'soil_moisture_7_to_28cm_mean',
    'temperature_2m_max',
    'wind_speed_10m_max',
    'rain_24h',
    'rain_48h',
    'rain_72h',
    'soil_saturation_index',
]

print(f'Artifacts will be saved to: {os.path.abspath(ARTIFACTS_DIR)}')
print('\nConfiguration:')
for k, v in CONFIG.items():
    print(f'  {k:<20}: {v}')

---

## 3. Data Ingestion and Validation

The raw CSV is loaded with explicit date parsing and immediately sorted by district and date to guarantee chronological ordering within each district time series. Initial validation checks cover data shape, column dtypes, missing values, and duplicate rows.

In [ ]:
df_raw = pd.read_csv(DATASET_PATH, parse_dates=['date'])
df_raw = df_raw.sort_values(['district', 'date']).reset_index(drop=True)

print('=== Dataset Overview ===')
print(f'Shape          : {df_raw.shape}')
print(f'Date range     : {df_raw["date"].min().date()}  to  {df_raw["date"].max().date()}')
print(f'Districts      : {df_raw["district"].nunique()}  unique')
print(f'Provinces      : {df_raw["province"].nunique()}  unique')
print(f'Climatic zones : {df_raw["climatic_zone"].unique().tolist()}')

df_raw.head(3)

In [ ]:
print('=== Data Quality Report ===')
print('\nColumn dtypes:')
print(df_raw.dtypes.to_string())

missing = df_raw.isnull().sum()
print('\nMissing values per column:')
if missing.sum() > 0:
    print(missing[missing > 0].to_string())
else:
    print('  None detected.')

dup_count = df_raw.duplicated(subset=['date', 'district']).sum()
print(f'\nDuplicate (date, district) rows: {dup_count}')
print(f'\nFlood risk score range: [{df_raw[TARGET_COL].min():.2f}, {df_raw[TARGET_COL].max():.2f}]')
print('\nFlood category counts:')
print(df_raw['flood_category'].value_counts().to_string())

---

## 4. Data Cleaning

Given that the dataset contains no missing values, the cleaning step focuses on removing any duplicate (date, district) pairs, deriving base temporal columns that will be used in feature engineering, and validating that every district has a complete daily time series across the full 10-year window.

In [ ]:
df = df_raw.drop_duplicates(subset=['date', 'district']).copy()

df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear

df = df.sort_values(['district', 'date']).reset_index(drop=True)

records_per_district = df.groupby('district').size()
expected_days        = (df['date'].max() - df['date'].min()).days + 1

print(f'Records after deduplication : {len(df):,}')
print(f'Expected days per district  : {expected_days}')
print(f'\nRecords per district (min={records_per_district.min()}, max={records_per_district.max()}):')
print(records_per_district.sort_values(ascending=False).to_string())

---

## 5. Exploratory Data Analysis

### 5.1 Temporal Risk Dynamics

The flood risk score is shaped by two major monsoon systems: the **Southwest Monsoon** (May-September), which primarily affects the western and southern slopes, and the **Northeast Monsoon** (October-January), which dominates the eastern and northern dry zones. Understanding how risk concentrates by month and year is essential for validating the target variable seasonal structure before modeling.

In [ ]:
monthly_pivot = (
    df.groupby(['year', 'month'])[TARGET_COL]
    .mean()
    .reset_index()
    .pivot(index='month', columns='year', values=TARGET_COL)
)
monthly_avg = df.groupby('month')[TARGET_COL].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im = axes[0].imshow(monthly_pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
axes[0].set_xticks(range(len(monthly_pivot.columns)))
axes[0].set_xticklabels(monthly_pivot.columns, rotation=45)
axes[0].set_yticks(range(12))
axes[0].set_yticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[0].set_title('Mean Flood Risk Score by Month and Year')
plt.colorbar(im, ax=axes[0], label='Flood Risk Score', shrink=0.85)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(range(1, 13), monthly_avg.values, color='steelblue', edgecolor='white', width=0.75)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels)
axes[1].set_title('Mean Flood Risk Score by Calendar Month (2015-2024)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Mean Flood Risk Score')

plt.suptitle('Temporal Risk Dynamics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 District-Level Risk Profile

Not all districts carry equal flood exposure. Low-elevation coastal and riverside districts tend to register both higher mean scores and higher peak events. This spatial heterogeneity is precisely why the global model encodes district identity as a learned categorical feature rather than training separate models per district.

In [ ]:
district_profile = (
    df.groupby('district')[TARGET_COL]
    .agg(mean_score='mean', max_score='max', std_score='std')
    .sort_values('mean_score', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

colors_mean = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(district_profile)))
axes[0].barh(district_profile.index, district_profile['mean_score'],
             color=colors_mean, edgecolor='white')
axes[0].set_title('Mean Flood Risk Score by District')
axes[0].set_xlabel('Mean Flood Risk Score')
axes[0].invert_yaxis()

dist_by_max = district_profile.sort_values('max_score', ascending=False)
colors_max  = plt.cm.Reds(np.linspace(0.3, 0.9, len(dist_by_max)))
axes[1].barh(dist_by_max.index, dist_by_max['max_score'],
             color=colors_max, edgecolor='white')
axes[1].set_title('Maximum Flood Risk Score by District')
axes[1].set_xlabel('Maximum Flood Risk Score')
axes[1].invert_yaxis()

plt.suptitle('District-Level Flood Risk Profile', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/district_risk_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print(district_profile.to_string())

### 5.3 Class Imbalance Analysis

The `flood_category` label derived from threshold cuts on `flood_risk_score` exhibits severe imbalance. Critical emergency events comprise fewer than 0.1% of all records. While the LSTM targets the continuous score rather than the category directly, understanding this imbalance is important for interpreting model performance: standard RMSE can be misleading when extreme events drive the most societal impact.

In [ ]:
cat_counts = df['flood_category'].value_counts()
palette    = ['#27ae60', '#f39c12', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

wedge_props = {'edgecolor': 'white', 'linewidth': 2.5}
axes[0].pie(
    cat_counts.values,
    labels=[c.replace(' (', '\n(') for c in cat_counts.index],
    autopct='%1.2f%%', colors=palette,
    wedgeprops=wedge_props, startangle=90
)
axes[0].set_title('Flood Category Distribution')

bars = axes[1].bar(range(len(cat_counts)), cat_counts.values,
                   color=palette, edgecolor='white', width=0.6)
axes[1].set_xticks(range(len(cat_counts)))
axes[1].set_xticklabels([c.replace(' (', '\n(') for c in cat_counts.index], ha='center')
axes[1].set_ylabel('Count (log scale)')
axes[1].set_yscale('log')
axes[1].set_title('Absolute Record Count per Category')
for bar, count in zip(bars, cat_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.15,
                 f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Flood Severity Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Imbalance ratio relative to the rarest class (Critical Emergency):')
print((cat_counts / cat_counts.min()).round(1).to_string())

### 5.4 Feature Correlation Analysis

Understanding pairwise linear correlations between predictors and the target surface informs which features carry the most direct predictive signal. Precipitation-related variables dominate, as expected, while temperature and wind speed carry a negative relationship due to their association with dry, non-monsoon conditions.

In [ ]:
corr_cols = BASE_FEATURES + [TARGET_COL]
corr_mat  = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(
    corr_mat, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, square=True,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('Pearson correlations with flood_risk_score (sorted):')
print(corr_mat[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False).to_string())

### 5.5 Flood Risk Score Distribution

The target variable has a heavily right-skewed distribution concentrated near zero, reflecting the dominance of low-risk conditions. The long tail toward extreme scores represents the rare but high-impact flood events. This motivates the choice of Huber loss for training, which combines the robustness of MAE for outliers with the smooth gradient properties of MSE near zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET_COL], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Flood Risk Score Distribution (Full Range)')
axes[0].set_xlabel('Flood Risk Score')
axes[0].set_ylabel('Frequency')

high_risk = df[df[TARGET_COL] > 20][TARGET_COL]
axes[1].hist(high_risk, bins=60, color='tomato', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Distribution of Elevated Risk Events (Score > 20, n={len(high_risk):,})')
axes[1].set_xlabel('Flood Risk Score')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Flood Risk Score Descriptive Statistics:')
print(df[TARGET_COL].describe().to_string())

---

## 6. Feature Engineering

Raw climatological observations are insufficient to capture the memory structure that drives flood risk. The LSTM will learn temporal dependencies within a 30-day window, but explicit engineered features provide multi-scale context that accelerates learning and improves generalization:

- **Lag features**: direct value at 1, 3, 7, and 14 days prior, preserving the exact signal at each past timestep
- **Rolling aggregates**: 7-, 14-, and 30-day rolling mean and standard deviation of precipitation, soil saturation, and risk score
- **Momentum signals**: the difference between short-term and medium-term rolling means, quantifying whether conditions are intensifying or easing
- **Cyclical time encodings**: sine/cosine transformations of month and day-of-year to encode seasonality as continuous periodic features without artificial discontinuities at year boundaries
- **Categorical encodings**: district and climatic zone encoded as integer labels, allowing the model to learn district-specific risk offsets

In [ ]:
def engineer_features(data):
    out = data.copy().sort_values(['district', 'date'])

    for lag in [1, 3, 7, 14]:
        out[f'precip_lag_{lag}d']   = out.groupby('district')['precipitation_sum'].shift(lag)
        out[f'risk_lag_{lag}d']     = out.groupby('district')[TARGET_COL].shift(lag)
        out[f'soil_sat_lag_{lag}d'] = out.groupby('district')['soil_saturation_index'].shift(lag)

    for window in [7, 14, 30]:
        out[f'precip_roll_mean_{window}d'] = out.groupby('district')['precipitation_sum'].transform(
            lambda x: x.rolling(window, min_periods=1).mean())
        out[f'precip_roll_std_{window}d'] = out.groupby('district')['precipitation_sum'].transform(
            lambda x: x.rolling(window, min_periods=1).std().fillna(0))
        out[f'risk_roll_mean_{window}d'] = out.groupby('district')[TARGET_COL].transform(
            lambda x: x.rolling(window, min_periods=1).mean())
        out[f'soil_sat_roll_mean_{window}d'] = out.groupby('district')['soil_saturation_index'].transform(
            lambda x: x.rolling(window, min_periods=1).mean())

    out['precip_momentum_7d'] = out.groupby('district')['precipitation_sum'].transform(
        lambda x: x.rolling(7, min_periods=1).mean() - x.rolling(14, min_periods=2).mean()
    ).fillna(0)

    out['soil_saturation_trend'] = out.groupby('district')['soil_saturation_index'].transform(
        lambda x: x.diff().rolling(7, min_periods=1).mean()
    ).fillna(0)

    out['sin_month'] = np.sin(2 * np.pi * out['month'] / 12)
    out['cos_month'] = np.cos(2 * np.pi * out['month'] / 12)
    out['sin_doy']   = np.sin(2 * np.pi * out['day_of_year'] / 365)
    out['cos_doy']   = np.cos(2 * np.pi * out['day_of_year'] / 365)

    le_district = LabelEncoder()
    le_zone     = LabelEncoder()
    out['district_enc']      = le_district.fit_transform(out['district'])
    out['climatic_zone_enc'] = le_zone.fit_transform(out['climatic_zone'])

    return out, le_district, le_zone

df_eng, le_district, le_zone = engineer_features(df)
print(f'Columns before engineering : {df.shape[1]}')
print(f'Columns after engineering  : {df_eng.shape[1]}')
print(f'Net new features           : {df_eng.shape[1] - df.shape[1]}')

In [ ]:
ENGINEERED_FEATURES = (
    BASE_FEATURES
    + [f'precip_lag_{l}d'   for l in [1, 3, 7, 14]]
    + [f'risk_lag_{l}d'     for l in [1, 3, 7, 14]]
    + [f'soil_sat_lag_{l}d' for l in [1, 3, 7, 14]]
    + [f'precip_roll_mean_{w}d'   for w in [7, 14, 30]]
    + [f'precip_roll_std_{w}d'    for w in [7, 14, 30]]
    + [f'risk_roll_mean_{w}d'     for w in [7, 14, 30]]
    + [f'soil_sat_roll_mean_{w}d' for w in [7, 14, 30]]
    + ['precip_momentum_7d', 'soil_saturation_trend']
    + ['sin_month', 'cos_month', 'sin_doy', 'cos_doy']
    + ['district_enc', 'climatic_zone_enc']
)

df_clean = df_eng.dropna(subset=ENGINEERED_FEATURES + [TARGET_COL]).copy()

print(f'Total engineered features : {len(ENGINEERED_FEATURES)}')
print(f'Records after dropna      : {len(df_clean):,}')
print('\nFeature list:')
for i, f in enumerate(ENGINEERED_FEATURES, 1):
    print(f'  {i:02d}. {f}')

---

## 7. Preprocessing and Data Splitting

### 7.1 Time-Based Train / Validation / Test Split

Data are partitioned strictly by year to prevent any form of temporal data leakage:

| Split | Years | Purpose |
|---|---|---|
| Training | 2015 to 2021 | Model fitting and weight updates |
| Validation | 2022 | Hyperparameter tuning, early stopping |
| Test | 2023 to 2024 | Final unbiased performance evaluation |

All scalers are fitted exclusively on the training split and then applied identically to validation and test data, simulating the constraints of real deployment.

In [ ]:
train_mask = df_clean['year'] < CONFIG['val_year_start']
val_mask   = (df_clean['year'] >= CONFIG['val_year_start']) & (df_clean['year'] < CONFIG['test_year_start'])
test_mask  = df_clean['year'] >= CONFIG['test_year_start']

feature_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler  = MinMaxScaler(feature_range=(0, 1))

feature_scaler.fit(df_clean.loc[train_mask, ENGINEERED_FEATURES])
target_scaler.fit(df_clean.loc[train_mask, [TARGET_COL]])

df_scaled = df_clean.copy()
df_scaled[ENGINEERED_FEATURES] = feature_scaler.transform(df_clean[ENGINEERED_FEATURES])
df_scaled['target_scaled']     = target_scaler.transform(df_clean[[TARGET_COL]])

print('Split summary:')
print(f'  Training set   : {train_mask.sum():>7,} records  (2015 - {CONFIG["val_year_start"]-1})')
print(f'  Validation set : {val_mask.sum():>7,} records  ({CONFIG["val_year_start"]} - {CONFIG["test_year_start"]-1})')
print(f'  Test set       : {test_mask.sum():>7,} records  ({CONFIG["test_year_start"]} - 2024)')

### 7.2 Sequence Construction

The LSTM requires fixed-length input sequences. A sliding window of `sequence_length = 30` days is applied independently within each district sorted time series. For each window position `i`, the model receives observations from day `i - 30` through day `i - 1` as input, and outputs the scaled flood risk score for days `i` through `i + 6` (a 7-day forecast horizon).

Sequences are generated per district to prevent cross-district contamination at window boundaries, then concatenated into a single dataset for the global model.

In [ ]:
def create_sequences(data, feat_cols, target_col, seq_len, horizon):
    X_list, y_list = [], []
    for district in data['district'].unique():
        d      = data[data['district'] == district].sort_values('date')
        X_vals = d[feat_cols].values
        y_vals = d[target_col].values
        n      = len(d)
        for i in range(seq_len, n - horizon + 1):
            X_list.append(X_vals[i - seq_len: i])
            y_list.append(y_vals[i: i + horizon])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

SEQ_LEN = CONFIG['sequence_length']
HORIZON = CONFIG['forecast_horizon']

X_train, y_train = create_sequences(df_scaled[train_mask], ENGINEERED_FEATURES, 'target_scaled', SEQ_LEN, HORIZON)
X_val,   y_val   = create_sequences(df_scaled[val_mask],   ENGINEERED_FEATURES, 'target_scaled', SEQ_LEN, HORIZON)
X_test,  y_test  = create_sequences(df_scaled[test_mask],  ENGINEERED_FEATURES, 'target_scaled', SEQ_LEN, HORIZON)

print('Sequence array shapes:')
print(f'  X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}  |  y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}  |  y_test  : {y_test.shape}')
print(f'\nEach sample -> input ({SEQ_LEN} steps, {len(ENGINEERED_FEATURES)} features) -> output ({HORIZON} forecast steps)')

### 7.3 PyTorch Dataset and DataLoader

The NumPy arrays are wrapped in a custom `FloodRiskDataset` that implements PyTorch Dataset interface, enabling efficient batching, shuffling (training only), and parallel data loading through `DataLoader`.

In [ ]:
class FloodRiskDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = CONFIG['batch_size']

train_loader = DataLoader(FloodRiskDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=0)
val_loader   = DataLoader(FloodRiskDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0)
test_loader  = DataLoader(FloodRiskDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

---

## 8. Model Architecture

### 8.1 Design Rationale

The architecture is a **Global Bidirectional LSTM with Multi-Step Output**:

| Component | Purpose |
|---|---|
| **Bidirectional LSTM (Layer 1)** | Processes the 30-day input sequence in both forward and backward directions, capturing how conditions both build up to and recover from high-risk periods. Returns full sequence of hidden states. |
| **Dropout + Layer Normalization** | Regularizes after each recurrent layer to prevent overfitting on the dominant low-risk regime. |
| **Unidirectional LSTM (Layer 2)** | Compresses the sequential representation into a fixed-size context vector at the final timestep. |
| **Dense Bottleneck** | Two fully connected layers with ReLU activation further transform the context vector. |
| **Multi-Step Output (Sigmoid)** | A single Dense layer of width 7 produces all forecast horizons simultaneously. Sigmoid bounds outputs to [0, 1] matching the MinMax-scaled target. |

**Loss function**: Huber loss (delta = 1.0) is chosen for its balanced treatment of small residuals (squared loss) and large outliers from extreme flood events (linear loss), avoiding gradient explosion caused by pure MSE on rare catastrophic scores.

**Optimizer**: Adam with gradient clipping (clip_norm = 1.0) and L2 weight decay for additional regularization.

In [ ]:
class FloodRiskLSTM(nn.Module):
    def __init__(self, input_size, hidden1, hidden2, dense_hidden, forecast_h, dropout):
        super(FloodRiskLSTM, self).__init__()

        self.bilstm     = nn.LSTM(input_size=input_size, hidden_size=hidden1,
                                  num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1   = nn.Dropout(dropout)
        self.layernorm1 = nn.LayerNorm(hidden1 * 2)

        self.lstm2      = nn.LSTM(input_size=hidden1 * 2, hidden_size=hidden2,
                                  num_layers=1, batch_first=True, bidirectional=False)
        self.dropout2   = nn.Dropout(dropout)
        self.layernorm2 = nn.LayerNorm(hidden2)

        self.fc1        = nn.Linear(hidden2, dense_hidden)
        self.act1       = nn.ReLU()
        self.dropout3   = nn.Dropout(dropout / 2)

        self.fc2        = nn.Linear(dense_hidden, dense_hidden // 2)
        self.act2       = nn.ReLU()
        self.dropout4   = nn.Dropout(dropout / 2)

        self.output_layer = nn.Linear(dense_hidden // 2, forecast_h)
        self.output_act   = nn.Sigmoid()

    def forward(self, x):
        out1, _       = self.bilstm(x)
        out1          = self.layernorm1(self.dropout1(out1))
        out2, (hn, _) = self.lstm2(out1)
        context       = self.layernorm2(self.dropout2(hn.squeeze(0)))
        out           = self.dropout3(self.act1(self.fc1(context)))
        out           = self.dropout4(self.act2(self.fc2(out)))
        return self.output_act(self.output_layer(out))


INPUT_SIZE = len(ENGINEERED_FEATURES)

model = FloodRiskLSTM(
    input_size   = INPUT_SIZE,
    hidden1      = CONFIG['lstm_hidden_1'],
    hidden2      = CONFIG['lstm_hidden_2'],
    dense_hidden = CONFIG['dense_hidden'],
    forecast_h   = HORIZON,
    dropout      = CONFIG['dropout_rate']
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Device               : {DEVICE}')

---

## 9. Model Training

Training uses:
- **Huber loss** (smooth L1 with delta = 1.0)
- **Adam optimizer** with weight decay for L2 regularization
- **ReduceLROnPlateau** scheduler: halves the learning rate when validation loss stagnates for 7 epochs
- **Early stopping**: training halts if validation loss does not improve for 15 consecutive epochs, restoring the best weights from the checkpoint

In [ ]:
criterion = nn.HuberLoss(delta=1.0)
optimizer = Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=CONFIG['lr_factor'],
                               patience=CONFIG['lr_patience'], min_lr=CONFIG['min_lr'], verbose=True)


def train_epoch(model, loader, criterion, optimizer, clip_val):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip_val)
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            total_loss += criterion(model(X_batch), y_batch).item() * len(X_batch)
    return total_loss / len(loader.dataset)


history    = {'train_loss': [], 'val_loss': [], 'lr': []}
best_val   = float('inf')
patience_c = 0
best_state = None
BEST_MODEL_PATH = f'{ARTIFACTS_DIR}/best_model.pt'

print(f'Starting training for up to {CONFIG["epochs"]} epochs...')
print(f'Early stopping patience : {CONFIG["patience"]} epochs')
print('-' * 65)

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, CONFIG['grad_clip'])
    val_loss   = eval_epoch(model, val_loader, criterion)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)

    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        patience_c = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_state, BEST_MODEL_PATH)
    else:
        patience_c += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:>4}/{CONFIG["epochs"]}  |  '
              f'train: {train_loss:.6f}  |  '
              f'val: {val_loss:.6f}  |  '
              f'lr: {current_lr:.2e}  |  '
              f'patience: {patience_c}/{CONFIG["patience"]}')

    if patience_c >= CONFIG['patience']:
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

model.load_state_dict(best_state)
print(f'\nBest validation loss: {best_val:.6f}  (best weights restored)')

---

## 10. Training History Visualization

The learning curves reveal the model convergence behavior. A healthy training run shows both curves declining in parallel with the gap closing, rather than training loss continuing to drop while validation loss stagnates or rises. The generalization gap chart and learning rate trace show exactly when the scheduler triggered reductions and whether overfitting developed.

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Training Loss',   linewidth=2, color='steelblue')
axes[0].plot(epochs_ran, history['val_loss'],   label='Validation Loss', linewidth=2, color='tomato')
axes[0].set_title('Huber Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].legend()

ax_lr = axes[0].twinx()
ax_lr.plot(epochs_ran, history['lr'], color='gray', linewidth=1, linestyle='--', alpha=0.6)
ax_lr.set_ylabel('Learning Rate', color='gray')
ax_lr.tick_params(axis='y', labelcolor='gray')
ax_lr.set_yscale('log')

gap = [v - t for t, v in zip(history['train_loss'], history['val_loss'])]
axes[1].plot(epochs_ran, gap, color='darkorange', linewidth=2)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1].fill_between(epochs_ran, gap, 0, alpha=0.15, color='darkorange')
axes[1].set_title('Generalization Gap (Val Loss - Train Loss)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Gap')

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total epochs trained : {len(history["train_loss"])}')
print(f'Best val loss        : {min(history["val_loss"]):.6f}  at epoch {history["val_loss"].index(min(history["val_loss"])) + 1}')
print(f'Final learning rate  : {history["lr"][-1]:.2e}')

---

## 11. Model Evaluation

### 11.1 Inference and Inverse Transform

Model outputs are in the [0, 1] normalized scale. Before computing interpretable metrics, predictions and ground truth are inverse-transformed back to the original flood risk score range using the target scaler fitted on training data. Evaluation is performed on both the validation set and the held-out test set.

In [ ]:
def run_inference(model, loader):
    model.eval()
    preds_list, true_list = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds_list.append(model(X_batch.to(DEVICE)).cpu().numpy())
            true_list.append(y_batch.numpy())
    return np.vstack(preds_list), np.vstack(true_list)


def compute_metrics(y_true, y_pred, horizon, split_name):
    y_true_flat = y_true.flatten()
    y_pred_flat = np.clip(y_pred.flatten(), 0, None)

    overall = {
        'rmse': float(np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))),
        'mae' : float(mean_absolute_error(y_true_flat, y_pred_flat)),
        'r2'  : float(r2_score(y_true_flat, y_pred_flat)),
        'mape': float(np.mean(np.abs((y_true_flat - y_pred_flat) /
                                      np.clip(np.abs(y_true_flat), 1e-6, None))) * 100)
    }
    per_horizon = {}
    for h in range(horizon):
        yt = y_true[:, h]
        yp = np.clip(y_pred[:, h], 0, None)
        per_horizon[f'T+{h+1}'] = {
            'rmse': float(np.sqrt(mean_squared_error(yt, yp))),
            'mae' : float(mean_absolute_error(yt, yp)),
            'r2'  : float(r2_score(yt, yp))
        }

    print(f'\n--- {split_name} Set Metrics ---')
    print(f'  RMSE  : {overall["rmse"]:.4f}')
    print(f'  MAE   : {overall["mae"]:.4f}')
    print(f'  R2    : {overall["r2"]:.4f}')
    print(f'  MAPE  : {overall["mape"]:.2f}%')
    print('\n  Per-Horizon Breakdown:')
    for k, v in per_horizon.items():
        print(f'    {k}  ->  RMSE: {v["rmse"]:.4f}  |  MAE: {v["mae"]:.4f}  |  R2: {v["r2"]:.4f}')
    return {'overall': overall, 'per_horizon': per_horizon}


val_preds_sc,  val_true_sc  = run_inference(model, val_loader)
test_preds_sc, test_true_sc = run_inference(model, test_loader)

val_true  = target_scaler.inverse_transform(val_true_sc)
val_pred  = target_scaler.inverse_transform(val_preds_sc)
test_true = target_scaler.inverse_transform(test_true_sc)
test_pred = target_scaler.inverse_transform(test_preds_sc)

val_metrics  = compute_metrics(val_true,  val_pred,  HORIZON, 'Validation')
test_metrics = compute_metrics(test_true, test_pred, HORIZON, 'Test')

### 11.2 Per-Horizon Metric Comparison

A fundamental characteristic of multi-step forecasting is performance degradation with increasing horizon. The model is expected to produce sharper predictions at T+1 and T+2, with growing uncertainty at T+6 and T+7. Quantifying this degradation curve is essential for communicating the practical reliability of each forecast day to operational end-users.

In [ ]:
horizons   = [f'T+{h+1}' for h in range(HORIZON)]
test_rmses = [test_metrics['per_horizon'][h]['rmse'] for h in horizons]
test_maes  = [test_metrics['per_horizon'][h]['mae']  for h in horizons]
test_r2s   = [test_metrics['per_horizon'][h]['r2']   for h in horizons]
val_rmses  = [val_metrics['per_horizon'][h]['rmse']  for h in horizons]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

x     = np.arange(len(horizons))
width = 0.35
axes[0].bar(x - width/2, val_rmses,  width, label='Validation', color='steelblue', edgecolor='white')
axes[0].bar(x + width/2, test_rmses, width, label='Test',       color='tomato',    edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(horizons)
axes[0].set_title('RMSE per Forecast Horizon')
axes[0].set_ylabel('RMSE')
axes[0].legend()

axes[1].bar(horizons, test_maes, color='teal', edgecolor='white')
axes[1].set_title('MAE per Forecast Horizon (Test)')
axes[1].set_xlabel('Horizon')
axes[1].set_ylabel('MAE')

axes[2].bar(horizons, test_r2s, color='darkorange', edgecolor='white')
axes[2].axhline(0,   color='red',  linestyle='--', alpha=0.7, linewidth=1.2, label='R2 = 0')
axes[2].axhline(0.5, color='gray', linestyle=':',  alpha=0.7, linewidth=1.0, label='R2 = 0.5')
axes[2].set_title('R2 Score per Forecast Horizon (Test)')
axes[2].set_xlabel('Horizon')
axes[2].set_ylabel('R2')
axes[2].legend()

plt.suptitle('Multi-Step Forecast Evaluation Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/per_horizon_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.3 Actual vs. Predicted — Multi-Horizon Time Series

The following chart overlays the model predictions against the true flood risk scores for the first 300 test sequences across all 7 forecast horizons. It provides intuitive visual evidence of forecast quality and highlights where predictions lead or lag the actual signal.

In [ ]:
N_VIZ = 300

fig, axes = plt.subplots(HORIZON, 1, figsize=(16, 3.2 * HORIZON), sharex=True)

for h in range(HORIZON):
    yt     = test_true[:N_VIZ, h]
    yp     = np.clip(test_pred[:N_VIZ, h], 0, None)
    rmse_h = test_metrics['per_horizon'][f'T+{h+1}']['rmse']
    r2_h   = test_metrics['per_horizon'][f'T+{h+1}']['r2']

    axes[h].plot(yt, label='Actual',    color='steelblue', linewidth=1.5, alpha=0.9)
    axes[h].plot(yp, label='Predicted', color='tomato',    linewidth=1.5, linestyle='--', alpha=0.85)
    axes[h].fill_between(range(N_VIZ), yt, yp, alpha=0.08, color='gray')
    axes[h].set_ylabel(f'T+{h+1}', fontsize=9)
    axes[h].set_title(f'Horizon T+{h+1}  |  RMSE = {rmse_h:.3f}  |  R2 = {r2_h:.3f}', fontsize=10)
    if h == 0:
        axes[h].legend(loc='upper right', framealpha=0.85)

axes[-1].set_xlabel('Sample Index (Test Set)')
plt.suptitle('Actual vs. Predicted Flood Risk Score — 7-Day Forecast Horizon (Test Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/actual_vs_predicted_multistep.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.4 Residual Analysis

Residual diagnostics examine whether prediction errors are random (desirable) or exhibit systematic bias. A symmetric residual distribution centered at zero with no visible trend in the residuals-vs-predicted plot indicates an unbiased model. A funnel-shaped pattern would suggest heteroscedasticity, typically meaning the model underestimates high-risk events.

In [ ]:
y_true_flat = test_true.flatten()
y_pred_flat = np.clip(test_pred.flatten(), 0, None)
residuals   = y_true_flat - y_pred_flat

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[0].axvline(residuals.mean(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Mean={residuals.mean():.2f}')
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sample_idx = np.random.choice(len(y_pred_flat), min(3000, len(y_pred_flat)), replace=False)
axes[1].scatter(y_pred_flat[sample_idx], residuals[sample_idx],
                alpha=0.3, s=6, color='steelblue', rasterized=True)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs. Predicted Values')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual')

axes[2].scatter(y_true_flat[sample_idx], y_pred_flat[sample_idx],
                alpha=0.3, s=6, color='teal', rasterized=True)
lim = max(y_true_flat.max(), y_pred_flat.max())
axes[2].plot([0, lim], [0, lim], 'r--', linewidth=2, label='Perfect Fit')
axes[2].set_title('Actual vs. Predicted Scatter')
axes[2].set_xlabel('Actual Score')
axes[2].set_ylabel('Predicted Score')
axes[2].legend()

plt.suptitle('Residual Analysis (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Residual statistics:')
print(f'  Mean   : {residuals.mean():.4f}')
print(f'  Std    : {residuals.std():.4f}')
print(f'  Min    : {residuals.min():.4f}')
print(f'  Max    : {residuals.max():.4f}')
print(f'  Median : {np.median(residuals):.4f}')

### 11.5 Flood Alert Detection Rate

Beyond regression metrics, the practical value of an early warning system is measured by its ability to correctly identify genuine flood alert days (score >= 30) across the test period. The table below frames the classification problem (Normal vs. Alert) at each forecast horizon, quantifying False Alarm Rate and Missed Detection Rate: the two most operationally critical failure modes for an early warning system.

In [ ]:
ALERT_THRESHOLD = 30.0

print(f'Alert threshold : flood_risk_score >= {ALERT_THRESHOLD}')
print('-' * 80)
print(f'{"Horizon":<8} {"TP":>6} {"FP":>6} {"FN":>6} {"TN":>8}  {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 80)

for h_idx, h_label in enumerate(horizons):
    yt = test_true[:, h_idx]
    yp = np.clip(test_pred[:, h_idx], 0, None)

    true_alert = (yt >= ALERT_THRESHOLD).astype(int)
    pred_alert = (yp >= ALERT_THRESHOLD).astype(int)

    tp = int(((true_alert == 1) & (pred_alert == 1)).sum())
    fp = int(((true_alert == 0) & (pred_alert == 1)).sum())
    fn = int(((true_alert == 1) & (pred_alert == 0)).sum())
    tn = int(((true_alert == 0) & (pred_alert == 0)).sum())

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)

    print(f'{h_label:<8} {tp:>6} {fp:>6} {fn:>6} {tn:>8}  {precision:>10.3f} {recall:>8.3f} {f1:>8.3f}')

---

## 12. Model Persistence and Knowledge Artifacts

All assets required to reload and deploy the trained model without rerunning this notebook are saved to the `artifacts/` directory. A structured JSON knowledge document captures the complete experiment record: architecture specification, training configuration, split definitions, feature manifest, evaluation outcomes, and provenance metadata.

In [ ]:
torch.save(model.state_dict(), f'{ARTIFACTS_DIR}/flood_risk_lstm_weights.pt')

model_config = {
    'input_size'     : INPUT_SIZE,
    'hidden1'        : CONFIG['lstm_hidden_1'],
    'hidden2'        : CONFIG['lstm_hidden_2'],
    'dense_hidden'   : CONFIG['dense_hidden'],
    'forecast_h'     : HORIZON,
    'dropout'        : CONFIG['dropout_rate'],
    'feature_list'   : ENGINEERED_FEATURES,
    'target'         : TARGET_COL,
    'sequence_length': SEQ_LEN,
}
with open(f'{ARTIFACTS_DIR}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

with open(f'{ARTIFACTS_DIR}/feature_scaler.pkl', 'wb') as f:
    pickle.dump(feature_scaler, f)

with open(f'{ARTIFACTS_DIR}/target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

with open(f'{ARTIFACTS_DIR}/label_encoder_district.pkl', 'wb') as f:
    pickle.dump(le_district, f)

with open(f'{ARTIFACTS_DIR}/label_encoder_zone.pkl', 'wb') as f:
    pickle.dump(le_zone, f)

training_log = {
    'train_loss'          : history['train_loss'],
    'val_loss'            : history['val_loss'],
    'lr'                  : history['lr'],
    'total_epochs_trained': len(history['train_loss']),
    'best_val_loss'       : min(history['val_loss']),
    'best_epoch'          : int(history['val_loss'].index(min(history['val_loss']))) + 1,
}
with open(f'{ARTIFACTS_DIR}/training_log.json', 'w') as f:
    json.dump(training_log, f, indent=2)

knowledge_doc = {
    'project'     : 'Sri Lanka District Flood Risk Early Warning System',
    'created_at'  : datetime.now().isoformat(),
    'dataset': {
        'file'          : 'sri_lanka_flood_risk_modeled.csv',
        'period'        : '2015-01-01 to 2024-12-31',
        'districts'     : int(df['district'].nunique()),
        'total_records' : int(len(df_raw)),
        'features_raw'  : len(BASE_FEATURES),
        'features_total': len(ENGINEERED_FEATURES),
    },
    'architecture': {
        'model_class'          : 'FloodRiskLSTM',
        'type'                 : 'Global Bidirectional LSTM + Stacked LSTM',
        'input_shape'          : [CONFIG['sequence_length'], INPUT_SIZE],
        'output_shape'         : HORIZON,
        'bilstm_hidden'        : CONFIG['lstm_hidden_1'],
        'lstm2_hidden'         : CONFIG['lstm_hidden_2'],
        'dense_hidden'         : CONFIG['dense_hidden'],
        'dropout_rate'         : CONFIG['dropout_rate'],
        'output_activation'    : 'Sigmoid',
        'loss_function'        : 'Huber (delta=1.0)',
        'optimizer'            : 'Adam',
        'weight_decay'         : CONFIG['weight_decay'],
        'gradient_clipping'    : CONFIG['grad_clip'],
        'total_parameters'     : total_params,
        'trainable_parameters' : trainable_params,
    },
    'training': {
        'sequence_length'        : CONFIG['sequence_length'],
        'forecast_horizon'       : HORIZON,
        'batch_size'             : CONFIG['batch_size'],
        'max_epochs'             : CONFIG['epochs'],
        'epochs_trained'         : len(history['train_loss']),
        'best_epoch'             : int(history['val_loss'].index(min(history['val_loss']))) + 1,
        'initial_lr'             : CONFIG['learning_rate'],
        'final_lr'               : history['lr'][-1],
        'early_stopping_patience': CONFIG['patience'],
        'train_split'            : f'2015 to {CONFIG["val_year_start"]-1}',
        'val_split'              : f'{CONFIG["val_year_start"]} to {CONFIG["test_year_start"]-1}',
        'test_split'             : f'{CONFIG["test_year_start"]} to 2024',
    },
    'features': {
        'feature_list'   : ENGINEERED_FEATURES,
        'target'         : TARGET_COL,
        'scaling'        : 'MinMaxScaler(0, 1) fit on training split only',
        'lag_days'       : [1, 3, 7, 14],
        'rolling_windows': [7, 14, 30],
    },
    'evaluation': {
        'alert_threshold': ALERT_THRESHOLD,
        'validation'     : val_metrics,
        'test'           : test_metrics,
    },
    'artifacts': {
        'model_weights'          : 'flood_risk_lstm_weights.pt',
        'best_checkpoint'        : 'best_model.pt',
        'model_config'           : 'model_config.json',
        'feature_scaler'         : 'feature_scaler.pkl',
        'target_scaler'          : 'target_scaler.pkl',
        'label_encoder_district' : 'label_encoder_district.pkl',
        'label_encoder_zone'     : 'label_encoder_zone.pkl',
        'training_log'           : 'training_log.json',
        'knowledge_document'     : 'model_knowledge.json',
    }
}
with open(f'{ARTIFACTS_DIR}/model_knowledge.json', 'w') as f:
    json.dump(knowledge_doc, f, indent=2, default=str)

print('All artifacts saved.')
print(f'\nFiles in {ARTIFACTS_DIR}/:')
for fname in sorted(os.listdir(ARTIFACTS_DIR)):
    size_kb = os.path.getsize(f'{ARTIFACTS_DIR}/{fname}') / 1024
    print(f'  {fname:<48}  {size_kb:>8.1f} KB')

---

## 13. Model Reload Demonstration

This cell demonstrates the full inference pipeline starting from saved artifacts: loading the model configuration, weights, and scalers, then generating a 7-day forecast for an arbitrary test batch. This validates that the saved artifacts are complete and portable for downstream deployment.

In [ ]:
with open(f'{ARTIFACTS_DIR}/model_config.json', 'r') as f:
    saved_config = json.load(f)

loaded_model = FloodRiskLSTM(
    input_size   = saved_config['input_size'],
    hidden1      = saved_config['hidden1'],
    hidden2      = saved_config['hidden2'],
    dense_hidden = saved_config['dense_hidden'],
    forecast_h   = saved_config['forecast_h'],
    dropout      = saved_config['dropout']
).to(DEVICE)

loaded_model.load_state_dict(
    torch.load(f'{ARTIFACTS_DIR}/flood_risk_lstm_weights.pt', map_location=DEVICE)
)
loaded_model.eval()

with open(f'{ARTIFACTS_DIR}/target_scaler.pkl', 'rb') as f:
    loaded_target_scaler = pickle.load(f)

demo_X = torch.tensor(X_test[:5], dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    demo_preds_scaled = loaded_model(demo_X).cpu().numpy()

demo_preds_orig = loaded_target_scaler.inverse_transform(demo_preds_scaled)
demo_true_orig  = loaded_target_scaler.inverse_transform(y_test[:5])

demo_df = pd.DataFrame(
    np.round(demo_preds_orig, 3),
    columns=[f'T+{h+1} (pred)' for h in range(HORIZON)],
    index=[f'Sample {i+1}' for i in range(5)]
)
true_df = pd.DataFrame(
    np.round(demo_true_orig, 3),
    columns=[f'T+{h+1} (true)' for h in range(HORIZON)],
    index=[f'Sample {i+1}' for i in range(5)]
)

print('Predicted flood risk scores (re-loaded model):')
print(demo_df.to_string())
print('\nGround truth:')
print(true_df.to_string())
print('\nModel reload successful.')

---

## 14. Summary and Conclusions

### What Was Built

A production-oriented **Multi-Step Flood Risk Early Warning System** for Sri Lanka, delivering daily flood risk score forecasts for T+1 through T+7 days ahead across all 25 administrative districts using a single global model.

### Architecture Summary

| Component | Specification |
|---|---|
| Model type | Global Bidirectional LSTM + Stacked LSTM |
| Input window | 30 days of historical observations |
| Output | 7 simultaneous forecast steps |
| Input features | 46 engineered features per timestep |
| Districts covered | 25 (all provinces and climatic zones) |
| Temporal scope | 2015 – 2024 (10 years, daily granularity) |

### Key Technical Decisions

- **Bidirectional first layer**: captures pre-event build-up and post-event drainage dynamics simultaneously within the lookback window.
- **Huber loss**: balances gradient stability on the majority low-risk regime with sensitivity to rare high-score events.
- **Strict temporal split**: prevents any form of future information leakage by fitting all preprocessing transformations on training data only.
- **Global model with district encoding**: a single model generalizes across all 25 districts, making it extensible to new districts without retraining.
- **Cyclical time features**: sine/cosine encodings of month and day-of-year allow the model to learn monsoon seasonality without treating December 31 and January 1 as maximally different days.

### Operational Interpretation

- **T+1 forecasts** carry the highest confidence and are suitable for same-day emergency alerts.
- **T+3 to T+5** forecasts support pre-positioning of relief resources and proactive community notices.
- **T+6 and T+7** forecasts indicate directional trend and should be treated as probabilistic guidance rather than precise point estimates.
- Any predicted score exceeding 30 (Advisory threshold) warrants operational verification before escalating response.

### Potential Extensions

- **Uncertainty quantification**: Monte Carlo Dropout or Deep Ensemble to provide confidence intervals alongside point forecasts.
- **Spatial graph layer**: Graph Convolutional Network encoding district adjacency to capture cross-district hydrological flow.
- **Attention mechanism**: Temporal attention over the 30-day input window to make the model focus on specific lag days interpretable.
- **Online learning**: incremental fine-tuning on incoming real-time observations as new daily data arrives.